In [ ]:
from telethon import TelegramClient
import asyncio
API_ID =  # Replace with your actual API_ID (integer)
API_HASH = '' # Replace with your actual API_HASH (string)
PHONE =  # integer
async def main():
    client = TelegramClient('my_session_name_for_id_finding', API_ID, API_HASH)
    await client.start(phone=PHONE)
    print("Client Created and Started...")
    print("\nListing your recent dialogs (channels/groups/users):")
    count = 0
    async for dialog in client.iter_dialogs(limit=199): # jitna most recent and pinned channel ka info chahiye, abhi top 20 hai
        count += 1
        entity_type = "User"
        if dialog.is_group:
            entity_type = "Group"
        if dialog.is_channel:
            entity_type = "Channel"        
        print(f"{count}. Title: '{dialog.name}', Type: {entity_type}, ID: {dialog.id}")
    print("\nIdentify the channel from the list above.")
    print("The 'ID' is what you need for TARGET_CHANNELS.")
    print("For channels, this ID is often negative. Use the full negative number.")
    await client.disconnect()
if __name__ == '__main__':
         asyncio.run(main())
    

In [ ]:
pip install asyncio   1002286002947 1001233303577

In [ ]:
# --- START OF FILE gbot.txt ---
import json
import os
from dotenv import load_dotenv
load_dotenv()

import re
import time
import logging
import asyncio
import sys
from datetime import datetime, timedelta
import pandas as pd
import MetaTrader5 as mt5
from decimal import Decimal # Added for precise volume step calculation

from telethon import TelegramClient, events
from telethon.sessions import MemorySession

logging.basicConfig(
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    level=logging.INFO,
    filename='telegram_mt5_bot.log'
)
logger = logging.getLogger(__name__)

API_ID = os.getenv('API_ID')
API_HASH = os.getenv('API_HASH')
PHONE = os.getenv('PHONE')

_target_channel_env = os.getenv('TARGET_CHANNEL_ID_TO_LISTEN')
TARGET_CHANNEL_ID_TO_LISTEN =  # Default if not set or invalid
if _target_channel_env and _target_channel_env.lstrip('-').isdigit():
    TARGET_CHANNEL_ID_TO_LISTEN = int(_target_channel_env)
else:
    logger.warning(f"TARGET_CHANNEL_ID_TO_LISTEN from .env is invalid ('{_target_channel_env}'). Using default: {TARGET_CHANNEL_ID_TO_LISTEN}")


TARGET_TELETHON_LISTENER_CHANNELS = [TARGET_CHANNEL_ID_TO_LISTEN]
TARGET_CHANNEL_FOR_TRADING_IDENTIFIER = str(TARGET_CHANNEL_ID_TO_LISTEN)

logger.info(f"Telethon will listen for messages from channel ID: {TARGET_TELETHON_LISTENER_CHANNELS}")
logger.info(f"Trading logic will only process signals from channel identifier: {TARGET_CHANNEL_FOR_TRADING_IDENTIFIER}")

_mt5_login_env = os.getenv('MT5_LOGIN')
MT5_LOGIN = int(_mt5_login_env) if _mt5_login_env and _mt5_login_env.isdigit() else 0

MT5_PASSWORD = os.getenv('MT5_PASSWORD')
MT5_SERVER = os.getenv('MT5_SERVER', 'Exness-MT5Trial7')
MT5_PATH = os.getenv('MT5_PATH', r"C:\Program Files\MetaTrader 5\terminal64.exe")

DEFAULT_VOLUME = float(os.getenv('DEFAULT_VOLUME', '0.01'))
PARTIAL_CLOSE_VOLUME = float(os.getenv('PARTIAL_CLOSE_VOLUME', '0.01'))
DEFAULT_SL_PIPS_TARGET = int(os.getenv('DEFAULT_SL_PIPS_TARGET', '70'))
DEFAULT_TP_PIPS_TARGET = int(os.getenv('DEFAULT_TP_PIPS_TARGET', '120'))
MAX_RETRIES = int(os.getenv('MAX_RETRIES', '50'))
QUICK_ORDER_TIMEOUT_MINUTES = int(os.getenv('QUICK_ORDER_TIMEOUT_MINUTES', '4'))


class TelegramMT5Bot:
    def __init__(self):
        self.client = None
        self.mt5_connected = False
        self.symbol_info_cache = {}
        self.managed_positions = {}  # Initialize as empty
        self.processed_signal_ids = set()
        self.order_pending_details = None
        self._validate_config()
        self._load_managed_positions_state() # Load state on init

    def _load_managed_positions_state(self):
        """Loads the managed_positions state from a JSON file."""
        try:
            if os.path.exists('managed_positions_state.json'):
                with open('managed_positions_state.json', 'r') as f:
                    loaded_data = json.load(f)
                    # JSON keys are strings, convert ticket back to int
                    self.managed_positions = {int(k): v for k, v in loaded_data.items()}
                    logger.info(f"Loaded {len(self.managed_positions)} managed positions from state file.")
            else:
                logger.info("No state file found (managed_positions_state.json). Starting fresh.")
        except Exception as e:
            logger.error(f"Error loading managed_positions_state.json: {e}. Starting fresh.", exc_info=True)
            self.managed_positions = {} # Ensure it's reset on error

    def _save_managed_positions_state(self):
        """Saves the current managed_positions state to a JSON file."""
        try:
            # Convert integer keys (tickets) to strings for JSON compatibility
            data_to_save = {str(k): v for k, v in self.managed_positions.items()}
            with open('managed_positions_state.json', 'w') as f:
                json.dump(data_to_save, f, indent=4)
            logger.info(f"Saved {len(self.managed_positions)} managed positions to state file.")
        except Exception as e:
            logger.error(f"Error saving managed_positions_state.json: {e}", exc_info=True)

    def _validate_config(self):
        missing_vars = []
        if not API_ID: missing_vars.append("API_ID")
        if not API_HASH: missing_vars.append("API_HASH")
        if not PHONE: missing_vars.append("PHONE")
        if not MT5_LOGIN or MT5_LOGIN == 0: missing_vars.append("MT5_LOGIN")
        if not MT5_PASSWORD: missing_vars.append("MT5_PASSWORD")
        if not os.getenv('TARGET_CHANNEL_ID_TO_LISTEN'): missing_vars.append("TARGET_CHANNEL_ID_TO_LISTEN")


        if missing_vars:
            print("\n⚠️ WARNING: Missing required environment variables (checked by _validate_config):")
            for var in missing_vars: print(f"  - {var}")
            print("\nCreate a .env file with these variables. Example .env file:")
            print(r"""
API_ID=YourApiId
API_HASH=YourApiHash
PHONE=+YourPhoneNumber

MT5_LOGIN=YourMt5Login
MT5_PASSWORD=YourMt5Password
MT5_SERVER=YourMt5Server
MT5_PATH="C:\Program Files\MetaTrader 5\terminal64.exe"

DEFAULT_VOLUME=0.01
PARTIAL_CLOSE_VOLUME=0.01
DEFAULT_SL_PIPS_TARGET=70
DEFAULT_TP_PIPS_TARGET=120
MAX_RETRIES=50
QUICK_ORDER_TIMEOUT_MINUTES=4
TARGET_CHANNEL_ID_TO_LISTEN=-100xxxxxxxxxx
            """)
            print("Continuing with default values where possible (may lead to errors)...")

    def _get_volume_precision(self, volume_step: float) -> int:
        if volume_step <= 0:
            logger.warning(f"Invalid volume_step {volume_step}, defaulting to precision 2.")
            return 2
        try:
            exponent = Decimal(str(volume_step)).normalize().as_tuple().exponent
            if isinstance(exponent, int): return abs(exponent)
            else: logger.warning(f"Could not determine exponent for volume_step {volume_step}, defaulting to precision 2."); return 2
        except Exception as e:
            logger.error(f"Error calculating volume precision for step {volume_step}: {e}. Defaulting to 2.")
            return 2

    def connect_mt5(self):
        try:
            if mt5.terminal_info() is not None and mt5.account_info() and mt5.account_info().login == MT5_LOGIN:
                logger.info(f"MT5 already initialized and logged into account {MT5_LOGIN}.")
                self.mt5_connected = True
                return True
            if not mt5.initialize(path=MT5_PATH):
                logger.error(f"MT5 initialization failed. Error: {mt5.last_error()}"); return False
            if not MT5_LOGIN or MT5_LOGIN == 0 or not MT5_PASSWORD:
                logger.error("MT5 login credentials not set or invalid."); mt5.shutdown(); return False
            if not mt5.login(MT5_LOGIN, password=MT5_PASSWORD, server=MT5_SERVER):
                logger.error(f"MT5 login failed. Error: {mt5.last_error()}"); mt5.shutdown(); return False
            account_info = mt5.account_info()
            if account_info is None:
                logger.error(f"Failed to get account info after login. Error: {mt5.last_error()}"); mt5.shutdown(); return False
            logger.info(f"Connected to MT5 as {account_info.login}, Balance: {account_info.balance} {account_info.currency}")
            print(f"✅ Connected to MetaTrader 5: Account {account_info.login}")
            self.mt5_connected = True
            return True
        except Exception as e:
            logger.error(f"Exception connecting to MT5: {str(e)}", exc_info=True); return False

    def get_symbol_info(self, symbol):
        if symbol in self.symbol_info_cache: return self.symbol_info_cache[symbol]
        symbol_info_obj = mt5.symbol_info(symbol)
        if symbol_info_obj is None: logger.warning(f"Failed to get symbol info for '{symbol}'. Error: {mt5.last_error()}"); return None
        self.symbol_info_cache[symbol] = symbol_info_obj
        return symbol_info_obj

    def find_mt5_symbol(self, user_symbol_name: str) -> str | None:
        if not self.mt5_connected: return None
        all_mt5_symbols = mt5.symbols_get()
        if all_mt5_symbols is None: return None
        mt5_names = [s.name for s in all_mt5_symbols]
        if user_symbol_name in mt5_names: return user_symbol_name
        user_lower = user_symbol_name.lower()
        for name in mt5_names:
            if name.lower() == user_lower: return name
        if "xau" in user_lower or "gold" in user_lower:
            gold_variants = ["XAUUSD", "XAUUSDm", "GOLD", "XAU/USD"]
            for gv in gold_variants:
                if gv in mt5_names: return gv
                if gv.lower() in [n.lower() for n in mt5_names]: return [n for n in mt5_names if n.lower() == gv.lower()][0]
        logger.warning(f"Symbol '{user_symbol_name}' not directly found. Further matching might be needed.")
        return None

    def calculate_sl_tp(self, symbol, order_type, entry_price):
        symbol_info = self.get_symbol_info(symbol)
        if symbol_info is None: logger.error(f"Cannot get symbol info for {symbol} in calculate_sl_tp."); return None, None
        sl_pips_target_config, tp_pips_target_config = DEFAULT_SL_PIPS_TARGET, DEFAULT_TP_PIPS_TARGET
        sl_distance_units, tp_distance_units = float(sl_pips_target_config), float(tp_pips_target_config)
        point, digits = symbol_info.point, symbol_info.digits
        price_unit_multiplier = 0.0
        # logger.debug(f"Symbol {symbol}: Digits={digits}, Point={point}, StopsLevel={getattr(symbol_info, 'stops_level', 'N/A')}, Spread={getattr(symbol_info, 'spread', 'N/A')}")
        if "XAU" in symbol.upper() or "GOLD" in symbol.upper():
            if digits == 2: price_unit_multiplier = point * 10
            elif digits == 3: price_unit_multiplier = point * 100
            else: price_unit_multiplier = point * 10; logger.warning(f"Gold symbol {symbol} has {digits} digits. Using point*10 for multiplier.")
        elif "BTC" in symbol.upper() or "ETH" in symbol.upper(): price_unit_multiplier = 1.0
        else:
            if digits == 5 or (digits == 3 and 'JPY' in symbol.upper()): price_unit_multiplier = point * 10
            elif digits in [2, 3]: price_unit_multiplier = point * 10
            elif digits == 0 and point == 0.01: price_unit_multiplier = point
            else: price_unit_multiplier = point * 10; logger.warning(f"Using fallback (1 config pip = 10 points) for {symbol} with {digits} digits.")
        if price_unit_multiplier == 0.0: price_unit_multiplier = point; logger.error(f"Multiplier zero for {symbol}. Defaulting to point.")
        if price_unit_multiplier == 0.0: logger.error(f"Symbol.point also zero for {symbol}. Cannot calc SL/TP."); return None, None
        sl_distance_price, tp_distance_price = sl_distance_units * price_unit_multiplier, tp_distance_units * price_unit_multiplier
        min_stop_price_distance = getattr(symbol_info, 'stops_level', 0) * point
        if min_stop_price_distance > 0:
            # logger.debug(f"Broker's min stops_level for {symbol}: {symbol_info.stops_level} points ({min_stop_price_distance:.{digits}f} price units).")
            if sl_distance_price < min_stop_price_distance: sl_distance_price = min_stop_price_distance; logger.warning(f"Adjusted SL dist for {symbol} to broker min.")
            if tp_distance_price < min_stop_price_distance: tp_distance_price = min_stop_price_distance; logger.warning(f"Adjusted TP dist for {symbol} to broker min.")
        if sl_distance_price <= 0: sl_distance_price = 100 * point; logger.warning(f"SL dist for {symbol} <=0. Set to 100*point.")
        if tp_distance_price <= 0: tp_distance_price = 200 * point; logger.warning(f"TP dist for {symbol} <=0. Set to 200*point.")
        sl = round(entry_price - sl_distance_price if order_type == mt5.ORDER_TYPE_BUY else entry_price + sl_distance_price, digits)
        tp = round(entry_price + tp_distance_price if order_type == mt5.ORDER_TYPE_BUY else entry_price - tp_distance_price, digits)
        min_safe_sl = round(entry_price - min_stop_price_distance if min_stop_price_distance > 0 else entry_price - (100*point), digits)
        min_safe_tp = round(entry_price + min_stop_price_distance if min_stop_price_distance > 0 else entry_price + (200*point), digits)
        max_safe_sl = round(entry_price + min_stop_price_distance if min_stop_price_distance > 0 else entry_price + (100*point), digits)
        max_safe_tp = round(entry_price - min_stop_price_distance if min_stop_price_distance > 0 else entry_price - (200*point), digits)
        if order_type == mt5.ORDER_TYPE_BUY:
            if sl >= entry_price: sl = min_safe_sl
            if tp <= entry_price: tp = min_safe_tp
        else: # SELL
            if sl <= entry_price: sl = max_safe_sl
            if tp >= entry_price: tp = max_safe_tp
        logger.info(f"Calc SL/TP for {symbol}: Entry={entry_price:.{digits}f}, Type={'B' if order_type == 0 else 'S'}, Final SL={sl:.{digits}f}, Final TP={tp:.{digits}f}")
        return sl, tp

    async def _execute_partial_close(self, ticket, symbol, order_type, volume, tp_level, current_bid, current_ask):
        symbol_info = self.get_symbol_info(symbol)
        if not symbol_info: logger.error(f"No symbol info for {symbol} in partial close."); return False
        volume_precision = self._get_volume_precision(symbol_info.volume_step)
        volume_to_close = round(volume, volume_precision)
        if volume_to_close < symbol_info.volume_min:
            logger.warning(f"Partial close volume {volume_to_close} for {ticket} is less than min {symbol_info.volume_min}. Cannot close.")
            return False # Cannot close less than min volume

        close_request = {"action": mt5.TRADE_ACTION_DEAL, "symbol": symbol, "volume": volume_to_close,
                         "type": mt5.ORDER_TYPE_SELL if order_type == mt5.ORDER_TYPE_BUY else mt5.ORDER_TYPE_BUY,
                         "position": ticket, "price": current_bid if order_type == mt5.ORDER_TYPE_BUY else current_ask,
                         "deviation": 20, "magic": 234000, "comment": f"Bot: Partial TP {tp_level} Hit"}
        logger.info(f"Attempting partial close for ticket {ticket} at TP {tp_level} with volume {volume_to_close}. Request: {close_request}")
        close_result = mt5.order_send(close_request)
        if close_result and close_result.retcode == mt5.TRADE_RETCODE_DONE:
            logger.info(f"Partial close for position {ticket} at TP {tp_level} successful. Deal: {close_result.deal}"); return True
        else:
            logger.error(f"Partial close for position {ticket} at TP {tp_level} FAILED. RetCode: {close_result.retcode if close_result else 'N/A'} Comm: {close_result.comment if close_result else 'N/A'}"); return False

    async def _modify_position_sltp(self, ticket, symbol, new_sl, new_tp):
        if not self.mt5_connected or mt5.terminal_info() is None: logger.error(f"MT5 not connected. Cannot modify SL/TP for ticket {ticket}."); return False
        position_info_list = mt5.positions_get(ticket=ticket)
        if not position_info_list: logger.error(f"Cannot find position with ticket {ticket} to modify SL/TP."); return False
        current_position = position_info_list[0]
        symbol_info = self.get_symbol_info(symbol)
        if not symbol_info: logger.error(f"Cannot get symbol info for {symbol} to validate SL/TP modification for ticket {ticket}."); return False
        final_sl = round(float(new_sl), symbol_info.digits) if new_sl is not None and float(new_sl) != 0.0 else 0.0
        final_tp = round(float(new_tp), symbol_info.digits) if new_tp is not None and float(new_tp) != 0.0 else 0.0
        # logger.debug(f"Preparing to modify ticket {ticket}: Symbol={symbol}, New SL={final_sl}, New TP={final_tp}")
        modify_request = {"action": mt5.TRADE_ACTION_SLTP, "position": ticket, "symbol": symbol, "sl": final_sl, "tp": final_tp,
                          "magic": current_position.magic, "comment": f"Bot: SLTP Mod {ticket}"}
        # logger.info(f"Attempting to modify SL/TP for ticket {ticket}. Request: {modify_request}")
        modify_result = mt5.order_send(modify_request)
        if modify_result and modify_result.retcode == mt5.TRADE_RETCODE_DONE:
            logger.info(f"Successfully updated SL/TP for position {ticket} to SL={final_sl}, TP={final_tp}."); return True
        else:
            retcode = modify_result.retcode if modify_result else "N/A"
            comm = modify_result.comment if modify_result else "N/A"
            logger.warning(f"Failed to update SL/TP for position {ticket}. RetCode: {retcode}, Comm: {comm}. MT5 LastError: {mt5.last_error()}. Request: {modify_request}"); return False

    async def _close_position_by_ticket(self, ticket, symbol, order_type, volume):
        if not self.mt5_connected: return False
        tick = mt5.symbol_info_tick(symbol);
        if not tick: return False
        price = tick.bid if order_type == mt5.ORDER_TYPE_BUY else tick.ask
        request = {"action": mt5.TRADE_ACTION_DEAL, "symbol": symbol, "volume": volume,
                   "type": mt5.ORDER_TYPE_SELL if order_type == mt5.ORDER_TYPE_BUY else mt5.ORDER_TYPE_BUY,
                   "position": ticket, "price": price, "deviation": 20, "magic": 234000,
                   "comment": "Bot: Closing unmanaged/timed-out order"}
        result = mt5.order_send(request)
        if result and result.retcode == mt5.TRADE_RETCODE_DONE: logger.info(f"Successfully closed position {ticket}."); return True
        else: logger.error(f"Failed to close position {ticket}. Error: {result.comment if result else mt5.last_error()}"); return False

    def list_available_symbols(self):
        if not self.mt5_connected: print("Cannot list symbols, MT5 not connected."); return
        try:
            symbols = mt5.symbols_get()
            if symbols:
                print(f"\n🔍 Found {len(symbols)} symbols. First few:")
                for i, s in enumerate(symbols[:5]): print(f"  - {s.name} (Digits: {s.digits}, Spread: {s.spread}, VolStep: {s.volume_step})")
                if len(symbols) > 5: print("  ...")
                # ... (rest of symbol listing)
            else: print("No symbols found.")
        except Exception as e: print(f"Error listing symbols: {e}")

    async def execute_market_order(self, symbol_from_signal, order_type, volume, sl_override=None, tp_override=None):
        if not self.mt5_connected: logger.error("MT5 not connected. Order aborted."); return False, None
        actual_mt5_symbol = self.find_mt5_symbol(symbol_from_signal)
        if not actual_mt5_symbol: logger.error(f"Symbol '{symbol_from_signal}' not resolved. Order aborted."); return False, None
        symbol = actual_mt5_symbol
        symbol_info = self.get_symbol_info(symbol)
        if not symbol_info: logger.error(f"Could not get symbol info for '{symbol}'. Order aborted."); return False, None
        volume_precision = self._get_volume_precision(symbol_info.volume_step)
        volume_to_trade = round(float(volume), volume_precision)
        if volume_to_trade < symbol_info.volume_min: volume_to_trade = symbol_info.volume_min; logger.warning(f"Vol {volume} for {symbol} < min {symbol_info.volume_min}. Adjusted to min.")
        if volume_to_trade > symbol_info.volume_max: volume_to_trade = symbol_info.volume_max; logger.warning(f"Vol {volume} for {symbol} > max {symbol_info.volume_max}. Adjusted to max.")
        volume_to_trade = round(round(volume_to_trade / symbol_info.volume_step) * symbol_info.volume_step, volume_precision)
        logger.info(f"Volume {volume} adjusted to {volume_to_trade} to meet step {symbol_info.volume_step} for {symbol}.")
        if not symbol_info.visible:
            if not mt5.symbol_select(symbol, True): logger.error(f"Failed to enable symbol {symbol}. Error: {mt5.last_error()}"); return False, None
            time.sleep(0.5); symbol_info = self.get_symbol_info(symbol)
            if not symbol_info or not symbol_info.visible: logger.error(f"Symbol {symbol} still not visible."); return False, None
        if symbol_info.trade_mode != mt5.SYMBOL_TRADE_MODE_FULL: logger.error(f"Symbol {symbol} not full trading. Mode: {symbol_info.trade_mode}"); return False, None
        tick = mt5.symbol_info_tick(symbol)
        if not tick or tick.time == 0: logger.error(f"Invalid tick for {symbol}. Error: {mt5.last_error()}"); return False, None
        price = tick.ask if order_type == mt5.ORDER_TYPE_BUY else tick.bid
        if price <= 0: logger.error(f"Invalid price ({price}) for {symbol}. Order aborted."); return False, None
        sl_price, tp_price, using_overrides = 0.0, 0.0, False
        if sl_override is None and tp_override is None: logger.info(f"Placing quick order for {symbol} with initial SL=0.0, TP=0.0.")
        elif sl_override is not None and tp_override is not None:
            try:
                sl_price, tp_price = float(sl_override), float(tp_override); using_overrides = True
                logger.info(f"Using provided SL/TP overrides for {symbol}: SL={sl_price}, TP={tp_price}")
                # Basic validation against price, detailed validation (stops_level) is complex here, rely on broker rejection or later adjustment
                if order_type == mt5.ORDER_TYPE_BUY and ( (sl_price != 0.0 and sl_price >= price) or (tp_price != 0.0 and tp_price <= price) ): logger.error(f"Invalid SL/TP override for BUY."); return False, None
                if order_type == mt5.ORDER_TYPE_SELL and ( (sl_price != 0.0 and sl_price <= price) or (tp_price != 0.0 and tp_price >= price) ): logger.error(f"Invalid SL/TP override for SELL."); return False, None
            except ValueError: logger.error(f"Invalid SL/TP override values: SL='{sl_override}', TP='{tp_override}'."); return False, None
        else:
            sl_price, tp_price = self.calculate_sl_tp(symbol, order_type, price)
            if sl_price is None or tp_price is None: logger.error(f"Failed to calculate SL/TP for {symbol}. Order aborted."); return False, None
            logger.info(f"Using calculated SL/TP for {symbol}: SL={sl_price}, TP={tp_price}")
        request = {"action": mt5.TRADE_ACTION_DEAL, "symbol": symbol, "volume": volume_to_trade, "type": order_type, "price": price,
                   "sl": sl_price, "tp": tp_price, "deviation": 20, "magic": 234000,
                   "comment": f"Bot: {symbol_from_signal}{' Override' if using_overrides else ''}",
                   "type_time": mt5.ORDER_TIME_GTC, "type_filling": mt5.ORDER_FILLING_IOC}
        print(f"📊 Market Order: {symbol} {'BUY' if order_type==0 else 'SELL'}@{price:.{symbol_info.digits}f} V:{volume_to_trade} SL:{sl_price:.{symbol_info.digits}f} TP:{tp_price:.{symbol_info.digits}f}{' (Overrides)' if using_overrides else ''}")
        # logger.debug(f"Order request for {symbol}: {request}")
        for attempt in range(MAX_RETRIES):
            result = mt5.order_send(request)
            if result is None: logger.error(f"Order_send None. Att {attempt+1}. Err: {mt5.last_error()}"); time.sleep(1 if attempt < MAX_RETRIES -1 else 0); continue
            if result.retcode == mt5.TRADE_RETCODE_DONE:
                logger.info(f"Order Executed: {symbol} Deal:{result.deal} Order:{result.order} Price:{result.price} Vol:{result.volume}"); print(f"✅ Order Executed: {symbol} Deal:{result.deal}"); return True, result.order
            else:
                logger.warning(f"Order Failed. Att {attempt+1}. RetCode:{result.retcode} Comm:'{result.comment}'")
                if result.retcode == mt5.TRADE_RETCODE_REQUOTE:
                    new_tick = mt5.symbol_info_tick(symbol)
                    if new_tick and new_tick.time > 0: request["price"] = new_tick.ask if order_type == mt5.ORDER_TYPE_BUY else new_tick.bid
                    else: logger.error("Could not get new tick for requote."); break
                    if not using_overrides and not (sl_override is None and tp_override is None):
                        sl_new, tp_new = self.calculate_sl_tp(symbol, order_type, request["price"])
                        if sl_new is None or tp_new is None: logger.error("Failed to recalc SL/TP on requote."); return False, None
                        request["sl"], request["tp"] = sl_new, tp_new; logger.info(f"Requote. New Px:{request['price']:.{symbol_info.digits}f}, SL:{sl_new:.{symbol_info.digits}f}, TP:{tp_new:.{symbol_info.digits}f}")
                    else: logger.info(f"Requote. New Px:{request['price']:.{symbol_info.digits}f}. Keeping orig SL/TP.")
                elif result.retcode in [mt5.TRADE_RETCODE_CONNECTION, mt5.TRADE_RETCODE_TIMEOUT, mt5.TRADE_RETCODE_SERVER_BUSY]: time.sleep(2)
                elif result.retcode == mt5.TRADE_RETCODE_INVALID_STOPS and using_overrides: logger.error(f"Invalid Stops with overrides SL={sl_price}, TP={tp_price}."); return False, None
                elif attempt < MAX_RETRIES - 1: time.sleep(1)
                else: logger.error(f"Order failed after all retries. Code: {result.retcode}"); return False, None
        return False, None

    async def process_message(self, message_text_original, chat_identifier=None):
        try:
            if chat_identifier != TARGET_CHANNEL_FOR_TRADING_IDENTIFIER: return
            message_text_cleaned = ''.join(char for char in message_text_original if 32 <= ord(char) <= 126).strip()
            if not message_text_cleaned: return
            logger.info(f"Processing msg from '{chat_identifier}': '{message_text_cleaned}'")
            current_signal_id = hash(message_text_cleaned)
            pattern_gold_multi_tp = re.compile(
                r"^\s*GOLD\s+(Buy|Sell)\s*"
                r"(?:(?:@|at|entry|price|zone|-)\s*[\d.]+(?:\s*-\s*[\d.]+)?)?\s*" # Optional entry price part
                r"TP\s*=\s*([\d.]+)\s*TP\s*=\s*([\d.]+)\s*TP\s*=\s*([\d.]+)\s*"
                r"STOP\s+LOSS\s*([\d.]+)\s*$", re.IGNORECASE | re.DOTALL)
            match_gold_multi_tp = pattern_gold_multi_tp.search(message_text_cleaned)

            if match_gold_multi_tp:
                current_signal_id = hash(message_text_cleaned) # Moved here to be available for all paths

                action_str_detailed = match_gold_multi_tp.group(1).upper()
                tps_str = [match_gold_multi_tp.group(2), match_gold_multi_tp.group(3), match_gold_multi_tp.group(4)]
                sl_str = match_gold_multi_tp.group(5)
                sl_detailed = float(sl_str)
                tps_from_signal_ordered = [float(tp) for tp in tps_str]
                order_type_detailed = mt5.ORDER_TYPE_BUY if action_str_detailed == "BUY" else mt5.ORDER_TYPE_SELL
                logger.info(f"Matched DETAILED GOLD: Action={action_str_detailed}, TPs={tps_from_signal_ordered}, SL={sl_detailed}. Signal ID: {current_signal_id}")

                # Check if this signal ID has already fully processed a position
                # A position is fully processed if it's in managed_positions, not awaiting details, and has this signal ID.
                for ticket_iter, pos_data_iter in self.managed_positions.items():
                    if pos_data_iter.get('processed_detailed_signal_id') == current_signal_id and \
                       not pos_data_iter.get('is_quick_order_awaiting_details'):
                        logger.info(f"Detailed GOLD signal ID {current_signal_id} already fully processed for managed position {ticket_iter}. Ignoring.")
                        return

                target_ticket, target_position_data, is_from_pending_details = None, None, False
                
                # 1. Check if it matches self.order_pending_details
                if self.order_pending_details:
                    pending_data = self.order_pending_details
                    if pending_data['symbol'].upper().startswith("XAUUSD") and pending_data['order_type'] == order_type_detailed:
                        if pending_data.get('processed_detailed_signal_id') == current_signal_id:
                             logger.info(f"Detailed signal ID {current_signal_id} already attempted for pending order {pending_data['ticket']} and it's still pending. Ignoring.")
                             return

                        target_ticket = pending_data['ticket']
                        live_pos_list = mt5.positions_get(ticket=target_ticket)
                        if not live_pos_list: 
                            logger.error(f"Pending order {target_ticket} not found live. Clearing pending details.")
                            self.order_pending_details = None
                            # Don't return yet, allow it to become a new order if this was the only match attempt
                        else:
                            live_pos = live_pos_list[0]
                            target_position_data = {
                                'symbol': pending_data['symbol'], 'initial_volume': live_pos.volume,
                                'current_volume': live_pos.volume, 'order_type': pending_data['order_type'],
                                'entry_price': live_pos.price_open, 'sl': 0, 'tps_to_hit': [],
                                'tps_hit_completed': [], 'current_sl': live_pos.sl,
                                'is_quick_order_awaiting_details': True
                            }
                            is_from_pending_details = True
                            logger.info(f"Detailed signal for pending quick order {target_ticket}.")
                
                # 2. If not matched pending (or pending disappeared), check timed-out quick orders in managed_positions
                if not target_ticket: 
                    for tid, p_data in list(self.managed_positions.items()):
                        if p_data['symbol'].upper().startswith("XAUUSD") and \
                           p_data['order_type'] == order_type_detailed and \
                           p_data.get('is_quick_order_awaiting_details'):
                            if p_data.get('processed_detailed_signal_id') == current_signal_id:
                                logger.info(f"Detailed signal ID {current_signal_id} already attempted for managed quick order {tid}. Ignoring this specific match.")
                                continue # Check if other quick orders match

                            target_ticket, target_position_data = tid, p_data
                            logger.info(f"Detailed signal for timed-out quick order {target_ticket}.")
                            break 
                
                # --- Decision Point ---
                if target_ticket and target_position_data:
                    # --- BRANCH A: Update existing quick order (pending or timed-out) ---
                    logger.info(f"Processing detailed signal ID {current_signal_id} for existing quick order {target_ticket}.")
                    
                    # --- This is the original logic for updating a quick order ---
                    # self.processed_signal_ids.add(current_signal_id) # Original placement
                    
                    actual_symbol = self.find_mt5_symbol(target_position_data['symbol'])
                    if not actual_symbol:
                        logger.error(f"Cannot resolve symbol {target_position_data['symbol']} for {target_ticket}.")
                        if is_from_pending_details: self.order_pending_details = None
                        return
                    
                    symbol_info = self.get_symbol_info(actual_symbol)
                    if not symbol_info:
                        logger.error(f"Cannot get symbol_info for {actual_symbol}.")
                        if is_from_pending_details: self.order_pending_details = None
                        return

                    entry_price = target_position_data['entry_price']
                    sl_is_invalid = not ((action_str_detailed == "BUY" and sl_detailed < entry_price) or \
                                       (action_str_detailed == "SELL" and sl_detailed > entry_price) or \
                                       sl_detailed == 0.0)
                    if sl_is_invalid:
                        logger.error(f"SL {sl_detailed} invalid for {action_str_detailed} order {target_ticket} entry {entry_price}.")
                        if is_from_pending_details: self.order_pending_details = None
                        # Potentially close the quick order if SL is bad
                        # await self._close_position_by_ticket(target_ticket, actual_symbol, order_type_detailed, target_position_data['current_volume'])
                        return

                    volume_precision = self._get_volume_precision(symbol_info.volume_step)
                    partial_unit_vol = round(PARTIAL_CLOSE_VOLUME, volume_precision)
                    volume_closed_now, tps_hit_now = 0.0, []
                    
                    live_pos_check = mt5.positions_get(ticket=target_ticket)
                    if not live_pos_check:
                        logger.warning(f"Position {target_ticket} disappeared before applying detailed signal.")
                        if target_ticket in self.managed_positions: del self.managed_positions[target_ticket]
                        if is_from_pending_details: self.order_pending_details = None
                        return
                    current_live_vol = round(live_pos_check[0].volume, volume_precision)
                    target_position_data['current_volume'] = current_live_vol # Update with live volume

                    tick_now = mt5.symbol_info_tick(actual_symbol)
                    if not tick_now:
                        logger.error(f"No tick for {actual_symbol} for immediate TP check on quick order {target_ticket}.")
                        # Proceed to SL/TP modification without immediate TP check if tick fails
                    else:
                        market_bid, market_ask = tick_now.bid, tick_now.ask
                        sorted_sig_tps = sorted(tps_from_signal_ordered) if action_str_detailed == "BUY" else sorted(tps_from_signal_ordered, reverse=True)
                        for tp_lvl in sorted_sig_tps:
                            tp_passed = (action_str_detailed == "BUY" and market_ask >= tp_lvl) or \
                                        (action_str_detailed == "SELL" and market_bid <= tp_lvl)
                            if tp_passed:
                                remaining_vol_to_manage = target_position_data['current_volume'] # Use updated live volume
                                if remaining_vol_to_manage >= partial_unit_vol and partial_unit_vol >= symbol_info.volume_min:
                                    logger.info(f"TP {tp_lvl} for {target_ticket} passed. Queuing {partial_unit_vol} for immediate close.")
                                    if await self._execute_partial_close(target_ticket, actual_symbol, target_position_data['order_type'], partial_unit_vol, tp_lvl, market_bid, market_ask):
                                        tps_hit_now.append(tp_lvl)
                                        # Update current_volume in target_position_data after successful partial close
                                        live_pos_after_partial = mt5.positions_get(ticket=target_ticket)
                                        if live_pos_after_partial:
                                            target_position_data['current_volume'] = round(live_pos_after_partial[0].volume, volume_precision)
                                        else:
                                            target_position_data['current_volume'] = 0.0
                                            logger.info(f"Pos {target_ticket} fully closed by immediate partials."); break
                                        await asyncio.sleep(0.5) 
                                    else: logger.error(f"Failed immediate partial close for TP {tp_lvl} on {target_ticket}. Stopping."); break
                                else: logger.info(f"TP {tp_lvl} passed, but not enough vol ({remaining_vol_to_manage}) for partial {partial_unit_vol} or partial too small. Stop immediate closes."); break
                            else: break 

                    if not isinstance(target_position_data.get('tps_hit_completed'), list): target_position_data['tps_hit_completed'] = []
                    for tp_h in tps_hit_now:
                        if tp_h not in target_position_data['tps_hit_completed']: target_position_data['tps_hit_completed'].append(tp_h)
                    
                    final_tp_mod = 0.0
                    remaining_tps_sig = [tp for tp in tps_from_signal_ordered if tp not in target_position_data['tps_hit_completed']]
                    
                    current_market_price_for_tp_check = tick_now.ask if action_str_detailed == "BUY" else tick_now.bid if tick_now else entry_price

                    if target_position_data['current_volume'] > 1e-8: 
                        if remaining_tps_sig:
                            valid_future_tps = []
                            if action_str_detailed == "BUY": valid_future_tps = sorted([tp for tp in remaining_tps_sig if tp > current_market_price_for_tp_check and (sl_detailed == 0.0 or tp > sl_detailed)])
                            else: valid_future_tps = sorted([tp for tp in remaining_tps_sig if tp < current_market_price_for_tp_check and (sl_detailed == 0.0 or tp < sl_detailed)], reverse=True)
                            
                            if valid_future_tps: final_tp_mod = valid_future_tps[-1] 
                            elif remaining_tps_sig : logger.warning(f"No remaining signal TPs currently achievable for {target_ticket}. Broker TP will be 0.0.")
                        else: logger.info(f"All signal TPs for {target_ticket} hit/passed. Broker TP will be 0.0.")
                        
                        logger.info(f"Modifying remaining pos {target_ticket} (Vol: {target_position_data['current_volume']}): SL={sl_detailed}, Final Broker TP={final_tp_mod}")
                        if await self._modify_position_sltp(target_ticket, actual_symbol, sl_detailed, final_tp_mod):
                            self.managed_positions[target_ticket] = {
                                'symbol': actual_symbol, 
                                'initial_volume': target_position_data.get('initial_volume', target_position_data['current_volume']), # Use initial if available
                                'current_volume': target_position_data['current_volume'], 
                                'order_type': target_position_data['order_type'],
                                'entry_price': entry_price, 'sl': sl_detailed, 
                                'tps_to_hit': remaining_tps_sig,
                                'tps_hit_completed': target_position_data['tps_hit_completed'], 
                                'partial_close_volume': partial_unit_vol,
                                'current_sl': sl_detailed, 
                                'is_quick_order_awaiting_details': False, # Crucial: now fully managed
                                'processed_detailed_signal_id': current_signal_id # Mark which signal processed it
                            }
                            self.processed_signal_ids.add(current_signal_id) # Successfully processed
                            logger.info(f"Pos {target_ticket} updated in managed_positions: {self.managed_positions[target_ticket]}")
                            if is_from_pending_details: self.order_pending_details = None # Clear pending once successfully managed
                        else: 
                            logger.error(f"Failed to modify SL/TP for {target_ticket} with detailed signal.")
                            # Do not add to processed_signal_ids if modification failed, allow retry
                    elif target_ticket in self.managed_positions: 
                        logger.info(f"Pos {target_ticket} no volume left. Removing from management.")
                        del self.managed_positions[target_ticket]
                        self.processed_signal_ids.add(current_signal_id) # Processed (by closing all volume)
                        if is_from_pending_details: self.order_pending_details = None

                    # if is_from_pending_details: self.order_pending_details = None # Already handled above on success
                    return # End of BRANCH A
                
                else: # No matching quick order found. This detailed signal is arriving first.
                    # --- BRANCH B: Create new order from detailed signal ---
                    logger.info(f"Detailed GOLD signal ID {current_signal_id} received first. No quick order to update. Attempting to place new order.")
                    
                    symbol_to_trade_detailed = "XAUUSDm" 
                    volume_detailed = DEFAULT_VOLUME
                    
                    actual_mt5_symbol_detailed = self.find_mt5_symbol(symbol_to_trade_detailed)
                    if not actual_mt5_symbol_detailed:
                        logger.error(f"Detailed-first: Symbol '{symbol_to_trade_detailed}' not resolved. Aborting signal {current_signal_id}.")
                        return

                    symbol_info_detailed = self.get_symbol_info(actual_mt5_symbol_detailed)
                    if not symbol_info_detailed:
                        logger.error(f"Detailed-first: Could not get symbol info for '{actual_mt5_symbol_detailed}'. Aborting signal {current_signal_id}.")
                        return

                    # Determine the furthest TP for initial order placement on broker
                    if action_str_detailed == "BUY": sorted_tps_for_broker = sorted(tps_from_signal_ordered)
                    else: sorted_tps_for_broker = sorted(tps_from_signal_ordered, reverse=True)
                    
                    initial_tp_for_broker_order = 0.0
                    if sorted_tps_for_broker: initial_tp_for_broker_order = sorted_tps_for_broker[-1] 
                    else: logger.warning(f"Detailed-first: No TPs in signal. Initial broker TP will be 0.0.")

                    logger.info(f"Placing new order from detailed signal: {action_str_detailed} {actual_mt5_symbol_detailed} Vol:{volume_detailed} SL:{sl_detailed} InitialBrokerTP:{initial_tp_for_broker_order}")
                    
                    success_new_detailed, order_ticket_obj_new = await self.execute_market_order(
                        actual_mt5_symbol_detailed, order_type_detailed, volume_detailed, 
                        sl_override=sl_detailed, tp_override=initial_tp_for_broker_order
                    )
                    # order_ticket_new is the actual ticket ID from the result object
                    order_ticket_new = order_ticket_obj_new # Assuming execute_market_order returns (bool, ticket_id)

                    if success_new_detailed and order_ticket_new:
                        live_pos_list_new = mt5.positions_get(ticket=order_ticket_new)
                        if not live_pos_list_new:
                            logger.error(f"Detailed-first: New order {order_ticket_new} placed but not found live. Signal {current_signal_id} might need manual check.")
                            return 
                        
                        live_pos_new = live_pos_list_new[0]
                        entry_price_new = live_pos_new.price_open
                        current_live_vol_new = live_pos_new.volume
                        volume_precision_new = self._get_volume_precision(symbol_info_detailed.volume_step)
                        partial_unit_vol_new = round(PARTIAL_CLOSE_VOLUME, volume_precision_new)

                        sl_is_invalid_new = not ((action_str_detailed == "BUY" and sl_detailed < entry_price_new) or \
                                               (action_str_detailed == "SELL" and sl_detailed > entry_price_new) or \
                                               sl_detailed == 0.0)
                        if sl_is_invalid_new:
                            logger.error(f"Detailed-first: SL {sl_detailed} invalid for new {action_str_detailed} order {order_ticket_new} (entry {entry_price_new}). Closing. Signal {current_signal_id}.")
                            await self._close_position_by_ticket(order_ticket_new, actual_mt5_symbol_detailed, order_type_detailed, current_live_vol_new)
                            return # Don't add to processed_signal_ids if SL is bad and order closed

                        logger.info(f"Detailed-first: New order {order_ticket_new} placed. Entry: {entry_price_new}, Vol: {current_live_vol_new}. Signal {current_signal_id}.")
                        
                        # Initial setup in managed_positions
                        self.managed_positions[order_ticket_new] = {
                            'symbol': actual_mt5_symbol_detailed, 'initial_volume': current_live_vol_new,
                            'current_volume': current_live_vol_new, 'order_type': order_type_detailed,
                            'entry_price': entry_price_new, 'sl': sl_detailed, 
                            'tps_to_hit': list(tps_from_signal_ordered), # Store all TPs from signal
                            'tps_hit_completed': [], 'partial_close_volume': partial_unit_vol_new,
                            'current_sl': sl_detailed, 'is_quick_order_awaiting_details': False,
                            'processed_detailed_signal_id': current_signal_id
                        }
                        # self.processed_signal_ids.add(current_signal_id) # Add after successful setup and immediate TP check

                        # --- Perform immediate TP check and partial close for the new order ---
                        tps_hit_now_new = [] # Renamed to avoid conflict
                        tick_now_new_order = mt5.symbol_info_tick(actual_mt5_symbol_detailed)
                        if not tick_now_new_order:
                            logger.error(f"Detailed-first: No tick for {actual_mt5_symbol_detailed} for immediate TP check on new order {order_ticket_new}.")
                        else:
                            market_bid_new, market_ask_new = tick_now_new_order.bid, tick_now_new_order.ask
                            # Use tps_from_signal_ordered, sort them appropriately for checking
                            sorted_sig_tps_new = sorted(tps_from_signal_ordered) if action_str_detailed == "BUY" else sorted(tps_from_signal_ordered, reverse=True)
                            
                            for tp_lvl_new in sorted_sig_tps_new:
                                if order_ticket_new not in self.managed_positions: break # Position might have been closed/removed
                                current_managed_pos_data = self.managed_positions[order_ticket_new]
                                tp_passed_new = (action_str_detailed == "BUY" and market_ask_new >= tp_lvl_new) or \
                                                (action_str_detailed == "SELL" and market_bid_new <= tp_lvl_new)
                                
                                if tp_passed_new:
                                    remaining_vol_new = current_managed_pos_data['current_volume']
                                    if remaining_vol_new >= partial_unit_vol_new and partial_unit_vol_new >= symbol_info_detailed.volume_min:
                                        logger.info(f"Detailed-first: Immediate TP {tp_lvl_new} for new order {order_ticket_new} passed. Queuing {partial_unit_vol_new} for close.")
                                        if await self._execute_partial_close(order_ticket_new, actual_mt5_symbol_detailed, order_type_detailed, partial_unit_vol_new, tp_lvl_new, market_bid_new, market_ask_new):
                                            tps_hit_now_new.append(tp_lvl_new)
                                            # Update managed_positions directly after successful partial close
                                            live_pos_after_partial_new = mt5.positions_get(ticket=order_ticket_new)
                                            current_vol_after_partial = round(live_pos_after_partial_new[0].volume, volume_precision_new) if live_pos_after_partial_new else 0.0
                                            self.managed_positions[order_ticket_new]['current_volume'] = current_vol_after_partial
                                            
                                            if tp_lvl_new not in self.managed_positions[order_ticket_new]['tps_hit_completed']:
                                                self.managed_positions[order_ticket_new]['tps_hit_completed'].append(tp_lvl_new)
                                            if tp_lvl_new in self.managed_positions[order_ticket_new]['tps_to_hit']:
                                                 self.managed_positions[order_ticket_new]['tps_to_hit'].remove(tp_lvl_new)
                                            await asyncio.sleep(0.5)
                                            if current_vol_after_partial <= 1e-8:
                                                logger.info(f"Detailed-first: New order {order_ticket_new} fully closed by immediate partials. Removing from management.")
                                                del self.managed_positions[order_ticket_new]
                                                break 
                                        else:
                                            logger.error(f"Detailed-first: Failed immediate partial close for TP {tp_lvl_new} on new order {order_ticket_new}. Stopping immediate closes.")
                                            break 
                                    else: 
                                        logger.info(f"Detailed-first: Immediate TP {tp_lvl_new} passed for new order {order_ticket_new}, but not enough vol ({remaining_vol_new}) for partial {partial_unit_vol_new} or partial too small. Marking TP.")
                                        if tp_lvl_new not in self.managed_positions[order_ticket_new]['tps_hit_completed']: self.managed_positions[order_ticket_new]['tps_hit_completed'].append(tp_lvl_new)
                                        if tp_lvl_new in self.managed_positions[order_ticket_new]['tps_to_hit']: self.managed_positions[order_ticket_new]['tps_to_hit'].remove(tp_lvl_new)
                                        break 
                                else: break # TP not passed
                        
                        # After immediate TP checks, if position still exists, update its SL/TP on broker if necessary
                        if order_ticket_new in self.managed_positions:
                            data_after_imm_tp = self.managed_positions[order_ticket_new]
                            final_broker_tp_after_imm = 0.0 # Default to 0 if no more TPs
                            remaining_tps_for_broker_mod = [tp for tp in data_after_imm_tp['tps_to_hit']] 

                            if data_after_imm_tp['current_volume'] > 1e-8 and remaining_tps_for_broker_mod:
                                current_market_for_broker_tp = market_ask_new if action_str_detailed == "BUY" else market_bid_new
                                if action_str_detailed == "BUY":
                                    valid_broker_tps = sorted([tp for tp in remaining_tps_for_broker_mod if tp > current_market_for_broker_tp and (sl_detailed == 0.0 or tp > sl_detailed)])
                                else: # SELL
                                    valid_broker_tps = sorted([tp for tp in remaining_tps_for_broker_mod if tp < current_market_for_broker_tp and (sl_detailed == 0.0 or tp < sl_detailed)], reverse=True)
                                if valid_broker_tps:
                                    final_broker_tp_after_imm = valid_broker_tps[-1] # Furthest valid remaining TP
                            
                            if data_after_imm_tp['current_volume'] > 1e-8: # Only modify if volume remains
                                # Check if current broker SL/TP matches what we want
                                live_pos_for_mod_check = mt5.positions_get(ticket=order_ticket_new)
                                if live_pos_for_mod_check and \
                                   (round(live_pos_for_mod_check[0].sl, symbol_info_detailed.digits) != round(sl_detailed, symbol_info_detailed.digits) or \
                                    round(live_pos_for_mod_check[0].tp, symbol_info_detailed.digits) != round(final_broker_tp_after_imm, symbol_info_detailed.digits)):
                                    logger.info(f"Detailed-first: Modifying broker SL/TP for {order_ticket_new} (Vol: {data_after_imm_tp['current_volume']}) after immediate TPs: SL={sl_detailed}, Final Broker TP={final_broker_tp_after_imm}")
                                    if not await self._modify_position_sltp(order_ticket_new, actual_mt5_symbol_detailed, sl_detailed, final_broker_tp_after_imm):
                                        logger.error(f"Detailed-first: Failed to modify SL/TP for new order {order_ticket_new} after immediate TP processing.")
                                else:
                                    logger.info(f"Detailed-first: Broker SL/TP for {order_ticket_new} already matches desired state after immediate TPs or no modification needed.")
                            elif order_ticket_new in self.managed_positions: # No volume left, ensure removal
                                logger.info(f"Detailed-first: New order {order_ticket_new} no volume left after immediate TPs. Removing from management.")
                                del self.managed_positions[order_ticket_new]
                        
                        self.processed_signal_ids.add(current_signal_id) # Mark signal as processed now
                        logger.info(f"Detailed-first: New order {order_ticket_new} fully set up. Signal {current_signal_id} marked processed.")

                    else: # execute_market_order failed
                        logger.error(f"Detailed-first: Failed to place new market order for detailed signal ID {current_signal_id}.")
                        # DO NOT add to processed_signal_ids, allow retry if signal comes again.
                    return # End of BRANCH B
            quick_signal_match = re.fullmatch(r"\s*(BUY|SELL)\s*", message_text_cleaned, re.IGNORECASE)
            if quick_signal_match:
                if self.order_pending_details:
                    logger.info(f"Quick signal '{message_text_cleaned}' received, but order {self.order_pending_details['ticket']} pending. Ignoring.")
                    self.processed_signal_ids.add(current_signal_id)
                    return
                
                self.processed_signal_ids.add(current_signal_id)
                action_str_quick = quick_signal_match.group(1).upper()
                logger.info(f"Signal ID {current_signal_id} matched QUICK '{action_str_quick}' signal.")
                symbol_to_trade_quick, volume_quick = "XAUUSDm", DEFAULT_VOLUME
                order_type_quick = mt5.ORDER_TYPE_BUY if action_str_quick == "BUY" else mt5.ORDER_TYPE_SELL
                logger.info(f"Processing QUICK Trade: Action='{action_str_quick}', Symbol='{symbol_to_trade_quick}', Vol={volume_quick}. No SL/TP.")
                
                actual_mt5_symbol_quick = self.find_mt5_symbol(symbol_to_trade_quick)
                if not actual_mt5_symbol_quick:
                    logger.error(f"Quick trade symbol '{symbol_to_trade_quick}' not resolved.")
                    if current_signal_id in self.processed_signal_ids:
                        self.processed_signal_ids.remove(current_signal_id)
                    return
                
                success_quick, ticket_id_quick = await self.execute_market_order(actual_mt5_symbol_quick, order_type_quick, volume_quick)
                if success_quick and ticket_id_quick:
                    logger.info(f"Quick order {ticket_id_quick} placed for {action_str_quick} {actual_mt5_symbol_quick}. Awaiting detailed signal.")
                    self.order_pending_details = {'ticket': ticket_id_quick, 'symbol': actual_mt5_symbol_quick, 'order_type': order_type_quick,
                                                  'volume': volume_quick, 'timestamp': datetime.now()}
                else:
                    logger.error(f"Quick order for {action_str_quick} {actual_mt5_symbol_quick} failed.")
                    if current_signal_id in self.processed_signal_ids:
                        self.processed_signal_ids.remove(current_signal_id)
                return
            
            if not match_gold_multi_tp and not quick_signal_match:
                 logger.info(f"Msg from '{chat_identifier}' (ID {current_signal_id}) did not match known trading signal. Msg: '{message_text_cleaned}'")
        except Exception as e:
            logger.error(f"Error in process_message for '{message_text_original[:100]}...': {e}", exc_info=True)
            if 'current_signal_id' in locals() and current_signal_id in self.processed_signal_ids:
                self.processed_signal_ids.remove(current_signal_id)

    async def check_pending_orders(self):
                if self.order_pending_details:
                    pending_data = self.order_pending_details
                    if datetime.now() - pending_data['timestamp'] > timedelta(minutes=QUICK_ORDER_TIMEOUT_MINUTES):
                        logger.warning(f"Order {pending_data['ticket']} pending detailed signal too long. Applying defaults.")
                        pos_info_list = mt5.positions_get(ticket=pending_data['ticket'])
                        if pos_info_list:
                            live_pos = pos_info_list[0]
                            actual_symbol = self.find_mt5_symbol(pending_data['symbol'])
                            if not actual_symbol: 
                                logger.error(f"Cannot resolve {pending_data['symbol']} for timed-out order {pending_data['ticket']}.")
                                self.order_pending_details = None
                                return 
                            default_sl, default_tp = self.calculate_sl_tp(actual_symbol, pending_data['order_type'], live_pos.price_open)
                            if default_sl is not None and default_tp is not None:
                                if await self._modify_position_sltp(pending_data['ticket'], actual_symbol, default_sl, default_tp):
                                    self.managed_positions[pending_data['ticket']] = {
                                        'symbol': actual_symbol, 'initial_volume': live_pos.volume, 'current_volume': live_pos.volume,
                                        'order_type': pending_data['order_type'], 'entry_price': live_pos.price_open, 'sl': default_sl,
                                        'tps_to_hit': [default_tp], 'tps_hit_completed': [], 
                                        'partial_close_volume': round(live_pos.volume * (PARTIAL_CLOSE_VOLUME / DEFAULT_VOLUME) if DEFAULT_VOLUME > 0 else live_pos.volume, self._get_volume_precision(self.get_symbol_info(actual_symbol).volume_step) if self.get_symbol_info(actual_symbol) else 2), # Pro-rata or full
                                        'current_sl': default_sl, 'is_quick_order_awaiting_details': True 
                                    }
                                    logger.info(f"Moved timed-out order {pending_data['ticket']} to managed with defaults and flag.")
                                else: logger.error(f"Failed to apply default SL/TP to timed-out order {pending_data['ticket']}.")
                            else: 
                                logger.error(f"Could not calc default SL/TP for timed-out {pending_data['ticket']}. Closing.")
                                await self._close_position_by_ticket(pending_data['ticket'], actual_symbol, pending_data['order_type'], live_pos.volume)
                        else: logger.warning(f"Timed-out order {pending_data['ticket']} not found live.")
                        self.order_pending_details = None
                if not self.managed_positions: return
                current_mt5_positions = {pos.ticket: pos for pos in mt5.positions_get() if pos.magic == 234000}
                
                for ticket in list(self.managed_positions.keys()): # Iterate over a copy
                    pos_data = self.managed_positions.get(ticket)
                    if not pos_data or pos_data.get('is_quick_order_awaiting_details'): 
                        live_pos_quick_check = current_mt5_positions.get(ticket)
                        if live_pos_quick_check:
                            if live_pos_quick_check.sl != 0.0: # Check SL if set
                                tick_qc = mt5.symbol_info_tick(live_pos_quick_check.symbol)
                                if tick_qc:
                                    if (live_pos_quick_check.type == mt5.ORDER_TYPE_BUY and tick_qc.bid <= live_pos_quick_check.sl) or \
                                    (live_pos_quick_check.type == mt5.ORDER_TYPE_SELL and tick_qc.ask >= live_pos_quick_check.sl):
                                        logger.info(f"Quick order {ticket} (awaiting details) SL hit. Removing from managed."); del self.managed_positions[ticket]
                        elif ticket in self.managed_positions: # If not live but in managed, remove
                            logger.info(f"Quick order {ticket} (awaiting details) not found live. Removing."); del self.managed_positions[ticket]
                        continue
                    live_position = current_mt5_positions.get(ticket)
                    if not live_position: logger.info(f"Managed pos {ticket} gone. Removing."); del self.managed_positions[ticket]; continue
                    
                    pos_data['current_volume'] = live_position.volume
                    pos_data['current_sl'] = live_position.sl
                    pos_data['current_tp'] = live_position.tp 

                    symbol, order_type = pos_data['symbol'], pos_data['order_type']
                    symbol_info = self.get_symbol_info(symbol)
                    if not symbol_info: logger.warning(f"No symbol_info for {symbol} in check_pending_orders."); continue
                    volume_precision = self._get_volume_precision(symbol_info.volume_step)
                    
                    tick = mt5.symbol_info_tick(symbol)
                    if not tick or tick.time == 0: logger.warning(f"No tick for {symbol} checking pos {ticket}."); continue
                    current_bid, current_ask = tick.bid, tick.ask

                    if pos_data['current_sl'] != 0.0: 
                        if (order_type == mt5.ORDER_TYPE_BUY and current_bid <= pos_data['current_sl']) or \
                        (order_type == mt5.ORDER_TYPE_SELL and current_ask >= pos_data['current_sl']):
                            logger.info(f"Managed pos {ticket} SL hit at {pos_data['current_sl']}. Removing."); del self.managed_positions[ticket]; continue
                    
                    tps_to_hit_sorted = sorted(pos_data.get('tps_to_hit', [])) if order_type == mt5.ORDER_TYPE_BUY else sorted(pos_data.get('tps_to_hit', []), reverse=True)
                while tps_to_hit_sorted:
                    next_tp_level = tps_to_hit_sorted[0]
                    tp_is_hit = (order_type == mt5.ORDER_TYPE_BUY and current_bid >= next_tp_level) or \
                                (order_type == mt5.ORDER_TYPE_SELL and current_ask <= next_tp_level)
                    
                    if tp_is_hit:
                        logger.info(f"Managed pos {ticket} TP {next_tp_level} hit. Vol: {pos_data['current_volume']}. TPs done: {len(pos_data.get('tps_hit_completed',[]))}.")
                        close_vol_unit = round(pos_data.get('partial_close_volume', PARTIAL_CLOSE_VOLUME), volume_precision)
                        vol_to_close_now = close_vol_unit
                        
                        if pos_data['current_volume'] < (close_vol_unit + symbol_info.volume_min) or pos_data['current_volume'] <= close_vol_unit :
                            vol_to_close_now = pos_data['current_volume'] 
                        
                        vol_to_close_now = round(vol_to_close_now, volume_precision)

                        if vol_to_close_now < symbol_info.volume_min and pos_data['current_volume'] >= symbol_info.volume_min :
                            vol_to_close_now = symbol_info.volume_min if pos_data['current_volume'] >= symbol_info.volume_min else 0.0
                        
                        if vol_to_close_now <= 1e-8 : 
                            logger.warning(f"TP {next_tp_level} hit, but no volume ({vol_to_close_now}) to close. Marking TP done."); 
                            if 'tps_hit_completed' not in pos_data: pos_data['tps_hit_completed'] = []
                            if next_tp_level not in pos_data['tps_hit_completed']: # Avoid duplicates
                                pos_data['tps_hit_completed'].append(next_tp_level)
                            if next_tp_level in pos_data['tps_to_hit']: pos_data['tps_to_hit'].remove(next_tp_level)
                            tps_to_hit_sorted.pop(0) # Remove from the sorted list being iterated
                            continue # Continue to check next TP in the sorted list
                        if await self._execute_partial_close(ticket, symbol, order_type, vol_to_close_now, next_tp_level, current_bid, current_ask):
                            await asyncio.sleep(0.5) 
                            live_pos_after_tp = mt5.positions_get(ticket=ticket)
                            if live_pos_after_tp: pos_data['current_volume'] = round(live_pos_after_tp[0].volume, volume_precision)
                            else: pos_data['current_volume'] = 0.0; logger.info(f"Pos {ticket} fully closed after TP {next_tp_level}.")
                            
                            if 'tps_hit_completed' not in pos_data: pos_data['tps_hit_completed'] = []
                            if next_tp_level not in pos_data['tps_hit_completed']: # Avoid duplicates
                                pos_data['tps_hit_completed'].append(next_tp_level)
                            if next_tp_level in pos_data['tps_to_hit']: pos_data['tps_to_hit'].remove(next_tp_level)
                            tps_to_hit_sorted.pop(0) # Remove from the sorted list being iterated
                            logger.info(f"Partial close for TP {next_tp_level} on {ticket} OK. Remaining vol: {pos_data['current_volume']}.")
                            new_sl_level = pos_data['current_sl'] 
                            entry_price = pos_data['entry_price']
                            
                            if not isinstance(pos_data.get('tps_hit_completed'), list): pos_data['tps_hit_completed'] = []
                            num_tps_actually_hit = len(pos_data['tps_hit_completed'])

                            if num_tps_actually_hit == 2: 
                                sl_buffer = symbol_info.point * 5 
                                candidate_be_sl = round(entry_price + sl_buffer if order_type == mt5.ORDER_TYPE_BUY else entry_price - sl_buffer, symbol_info.digits)
                                should_move_to_be = False
                                if pos_data['current_sl'] == 0.0: should_move_to_be = True
                                elif order_type == mt5.ORDER_TYPE_BUY and candidate_be_sl > pos_data['current_sl']: should_move_to_be = True
                                elif order_type == mt5.ORDER_TYPE_SELL and candidate_be_sl < pos_data['current_sl']: should_move_to_be = True
                                if should_move_to_be:
                                    new_sl_level = candidate_be_sl
                                    tp2_value = pos_data['tps_hit_completed'][1] if len(pos_data['tps_hit_completed']) > 1 else "TP2"
                                    logger.info(f"TP2 ({tp2_value}) hit. Moving SL to Break-Even for {ticket}: {new_sl_level}")
                                else: logger.info(f"TP2 hit for {ticket}. Candidate BE SL {candidate_be_sl} not an improvement. SL not changed.")
                            
                            if new_sl_level != pos_data['current_sl'] and pos_data['current_volume'] > 1e-8:
                                next_tp_for_rem_broker = 0.0 
                                current_market_ref_price_for_tp = current_bid if order_type == mt5.ORDER_TYPE_BUY else current_ask
                                if pos_data.get('tps_to_hit'): 
                                    valid_future_signal_tps = []
                                    if order_type == mt5.ORDER_TYPE_BUY:
                                        valid_future_signal_tps = sorted([tp for tp in pos_data['tps_to_hit'] if tp > current_market_ref_price_for_tp and (new_sl_level == 0.0 or tp > new_sl_level)])
                                        if valid_future_signal_tps: next_tp_for_rem_broker = valid_future_signal_tps[-1]
                                    else: 
                                        valid_future_signal_tps = sorted([tp for tp in pos_data['tps_to_hit'] if tp < current_market_ref_price_for_tp and (new_sl_level == 0.0 or tp < new_sl_level)], reverse=True)
                                        if valid_future_signal_tps: next_tp_for_rem_broker = valid_future_signal_tps[-1]
                                    if not valid_future_signal_tps: logger.info(f"No valid future signal TPs for {ticket}. Broker TP to 0.0.")
                                else: logger.info(f"All signal TPs processed for {ticket}. Broker TP to 0.0.")
                                if await self._modify_position_sltp(ticket, symbol, new_sl_level, next_tp_for_rem_broker): pos_data['current_sl'] = new_sl_level
                                else: logger.warning(f"Failed to update SL for {ticket} to {new_sl_level} (TP to {next_tp_for_rem_broker}).")
                            if pos_data['current_volume'] <= 1e-8: 
                                break # Break from TP checking (while) loop if position fully closed
                        else: 
                            logger.error(f"Failed partial close for TP {next_tp_level} on {ticket}."); 
                            break # Break from TP checking (while) loop
                    else: # This else corresponds to "if tp_is_hit:"
                        break
                if ticket in self.managed_positions: # Check if position still exists in management
                    if pos_data['current_volume'] <= 1e-8:
                        logger.info(f"Position {ticket} is fully closed. Removing from management."); 
                        del self.managed_positions[ticket]
                    elif not tps_to_hit_sorted and not pos_data.get('is_quick_order_awaiting_details'): 
                        if not pos_data.get('tps_to_hit'): # Check if the original list of TPs to hit is empty
                            logger.info(f"All signal TPs for {ticket} processed. Position now managed by its own SL/TP. Removing from active multi-TP management.")
                            del self.managed_positions[ticket]
    async def connect_telegram(self):
        if not API_ID or not API_HASH: logger.error("Telegram API_ID or API_HASH not set."); print("❌ No API_ID/HASH."); self.client=None; return False
        try:
            self.client = TelegramClient('gbot_session', API_ID, API_HASH)
            logger.info("TelegramClient created. Connecting..."); await self.client.connect()
            if not await self.client.is_user_authorized():
                logger.info("User not authorized. Sending code..."); await self.client.send_code_request(PHONE)
                try: code = await asyncio.to_thread(input, 'Enter Telegram code: '); await self.client.sign_in(PHONE, code); logger.info("Signed in.")
                except Exception as e: logger.error(f"Telegram sign-in failed: {e}", exc_info=True); await self.client.disconnect(); self.client=None; return False
            if not self.client or not self.client.is_connected(): logger.error("Failed to establish Telegram session."); self.client=None; return False
            me = await self.client.get_me(); logger.info(f"Connected to Telegram as: {me.username if me and me.username else me.id if me else 'N/A'}"); print("✅ Connected to Telegram.")
            @self.client.on(events.NewMessage(chats=TARGET_TELETHON_LISTENER_CHANNELS))
            async def handle_new_message(event):
                chat_id_str = str(event.chat_id) if hasattr(event, 'chat_id') else "UnknownChat"
                # logger.debug(f"Raw msg from '{chat_id_str}': {event.message.text[:150]}...")
                await self.process_message(event.message.text, chat_id_str)
            logger.info(f"Event handler registered for channels: {TARGET_TELETHON_LISTENER_CHANNELS}"); return True
        except Exception as e: logger.error(f"Failed to connect Telegram: {e}", exc_info=True); self.client=None; return False

    async def run(self):
        try:
            print("Connecting to MT5...");
            if not self.connect_mt5(): print("❌ Critical: Failed to connect to MT5. Bot cannot start."); return False
            print("✓ MT5 Connected."); self.list_available_symbols()

            # <<< NEW: Reconcile loaded positions after MT5 connection >>>
            await self._reconcile_loaded_positions_after_startup()

            print("Connecting to Telegram...");
            if not await self.connect_telegram(): print("❌ Critical: Failed to connect to Telegram. Bot cannot start."); return False
            if self.client is None: print("❌ Critical: Telegram client not initialized. Bot cannot run."); return False
            print(f"✓ Telegram Connected! Monitoring: {TARGET_TELETHON_LISTENER_CHANNELS}"); logger.info("Bot started successfully."); print("\nBot Running! (Ctrl+C to stop)\n")
            
            while True:
                if not self.mt5_connected or mt5.terminal_info() is None:
                    logger.warning("MT5 connection lost. Reconnecting...");
                    if not self.connect_mt5(): logger.error("Failed to reconnect MT5."); await asyncio.sleep(60); continue
                    else: logger.info("Reconnected to MT5.")
                await self.check_pending_orders(); await asyncio.sleep(5) # Check frequently
        except KeyboardInterrupt: logger.info("Bot stopped by user."); print("\nBot stopping...")
        except asyncio.CancelledError: logger.info("Bot run task cancelled."); print("\nBot task cancelled...")
        except Exception as e: logger.critical(f"Critical error in bot's main run loop: {e}", exc_info=True); print(f"❌ Critical error: {e}")
        finally:
            print("\nInitiating cleanup...");
            # <<< NEW: Save state before full shutdown >>>
            self._save_managed_positions_state()
            
            if self.client and self.client.is_connected(): 
                try:
                    logger.info("Disconnecting Telegram..."); await self.client.disconnect(); logger.info("Telegram disconnected.")
                except Exception as tg_e: logger.error(f"Error disconnecting telegram: {tg_e}")
            if self.mt5_connected: logger.info("Shutting down MT5..."); mt5.shutdown(); self.mt5_connected = False; logger.info("MT5 shut down.")
            print("Cleanup finished. Bot shutdown."); logger.info("Bot shutdown sequence complete.")
        return True

    async def _reconcile_loaded_positions_after_startup(self):
        """
        Compares loaded managed positions with live MT5 positions.
        Removes positions from management if they are no longer live.
        Updates live data like current volume, SL, TP for loaded positions.
        """
        if not self.mt5_connected:
            logger.warning("MT5 not connected. Cannot reconcile loaded positions.")
            return
        if not self.managed_positions: # No positions were loaded from file
            return

        logger.info(f"Reconciling {len(self.managed_positions)} loaded managed positions with live MT5 data...")
        live_mt5_positions = {pos.ticket: pos for pos in mt5.positions_get() if pos.magic == 234000}
        
        managed_tickets_to_remove = []
        for ticket, pos_data_from_file in list(self.managed_positions.items()): # Iterate copy
            live_pos_info = live_mt5_positions.get(ticket)

            if not live_pos_info:
                logger.info(f"Loaded managed position {ticket} is no longer live in MT5. Removing from management.")
                managed_tickets_to_remove.append(ticket)
                continue
            
            # Update essential live info from MT5 into the loaded position data
            self.managed_positions[ticket]['current_volume'] = live_pos_info.volume
            self.managed_positions[ticket]['current_sl'] = live_pos_info.sl # SL on broker
            # self.managed_positions[ticket]['current_tp'] = live_pos_info.tp # TP on broker (furthest)
            self.managed_positions[ticket]['entry_price'] = live_pos_info.price_open 
            logger.info(f"Reconciled loaded position {ticket}. Current Vol: {live_pos_info.volume}, SL: {live_pos_info.sl}, TPs to hit: {self.managed_positions[ticket].get('tps_to_hit')}")

        for ticket_to_remove in managed_tickets_to_remove:
            if ticket_to_remove in self.managed_positions:
                del self.managed_positions[ticket_to_remove]
        
        logger.info("Reconciliation of loaded positions complete.")

async def main_async_runner():
    if not os.path.exists('.env'): create_default_env_file(); print("\nIMPORTANT: .env created. Edit and restart."); return
    load_dotenv(override=True) 
    essential = ['API_ID', 'API_HASH', 'PHONE', 'MT5_LOGIN', 'MT5_PASSWORD', 'TARGET_CHANNEL_ID_TO_LISTEN']
    missing = [var for var in essential if not os.getenv(var)]
    mt5_login_str = os.getenv('MT5_LOGIN')
    mt5_login_invalid = not (mt5_login_str and mt5_login_str.isdigit() and int(mt5_login_str) != 0)
    target_ch_str = os.getenv('TARGET_CHANNEL_ID_TO_LISTEN')
    target_ch_invalid = not (target_ch_str and target_ch_str.lstrip('-').isdigit())
    if missing or mt5_login_invalid or target_ch_invalid:
        print("\n⚠️ Essential .env variables missing or invalid:")
        for var in missing: print(f"  - {var} is missing.")
        if mt5_login_invalid: print(f"  - MT5_LOGIN ('{mt5_login_str}') is invalid/zero.")
        if target_ch_invalid: print(f"  - TARGET_CHANNEL_ID_TO_LISTEN ('{target_ch_str}') is invalid.")
        print("Please check your .env file."); return
    bot = TelegramMT5Bot(); await bot.run()

def create_default_env_file():
    content = r"""# Telegram API credentials
API_ID=
API_HASH=
PHONE=
# MT5 Credentials
MT5_LOGIN=
MT5_PASSWORD=
MT5_SERVER=Exness-MT5Trial7
MT5_PATH="C:\Program Files\MetaTrader 5\terminal64.exe"
# Trading Parameters
DEFAULT_VOLUME=0.01
PARTIAL_CLOSE_VOLUME=0.01
DEFAULT_SL_PIPS_TARGET=70
DEFAULT_TP_PIPS_TARGET=120
MAX_RETRIES=50
QUICK_ORDER_TIMEOUT_MINUTES=4
# Telegram Channel to Listen To (MUST be a number, e.g., -100xxxxxxxxxx)
TARGET_CHANNEL_ID_TO_LISTEN=
"""
    try:
        with open('.env', 'w') as f: f.write(content)
        print("✅ Default .env file created. Please edit with your credentials.")
    except Exception as e: print(f"❌ Error creating .env file: {e}")

if __name__ == "__main__":
    try:
        try: import nest_asyncio; nest_asyncio.apply(); 
        except ImportError: pass 
        os.system('cls' if os.name == 'nt' else 'clear')
        print("="*50 + "\n Telegram to MetaTrader 5 Trading Bot\n" + "="*50 + "\nInitializing bot...\n")
        load_dotenv(override=True) 
        if os.getenv('DEBUG_MT5_BOT') == '1': print("🧪 DEBUG_MT5_BOT=1. Skipping main bot run.")
        else: asyncio.run(main_async_runner())
    except KeyboardInterrupt: print("\nGlobal script termination by user.")
    except Exception as e: logger.critical(f"Global script error: {e}", exc_info=True); print(f"❌ Global critical error: {e}")
    finally:
        print("\nBot shutdown sequence from __main__.")
        if mt5.terminal_info() is not None: mt5.shutdown() 
        print("Bot fully shutdown. Check 'telegram_mt5_bot.log'.\n" + "="*50)